# 02. ETL and PostgreSQL Ingestion

This notebook:

1. Loads the narrowed checkpoint from Notebook 01 (7,079,061 rows × 37 columns).
2. Creates the staging table `staging.stg_bts_flights` in a dedicated database, `bts_flights_db`.
3. Renames columns to snake_case and builds a SHA-256 `row_hash` surrogate key.
4. Loads the data in batches with `INSERT ... ON CONFLICT DO NOTHING`, so the load can be repeated safely.
5. Validates the load against the source DataFrame.

**Scope.** Getting clean, validated data into PostgreSQL. All analysis happens in Notebook 03.

**Prerequisites.** `checkpoints/df_narrowed_2024_bts.parquet` from Notebook 01, an empty PostgreSQL database called `bts_flights_db` (I created it in pgAdmin 4), and a `.env` file in the project root with `DB_USER`, `DB_PASSWORD`, `DB_HOST`, `DB_PORT` and `DB_NAME`.

# 1. Environment setup

`hashlib` builds the surrogate key, `python-dotenv` keeps credentials out of the notebook, and SQLAlchemy with psycopg2 handles the connection. For the bulk insert I use psycopg2's `execute_values`, which sends many rows per statement.

The notebook runs from the `notebooks/` folder, so the project root (where `.env` lives) is one level up. I resolve that path explicitly because `find_dotenv()` failed silently when the working directory didn't match.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import hashlib
import logging
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from psycopg2.extras import execute_values
from sqlalchemy import create_engine, text
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    logger.info(f"Connected to database: {result.scalar()}")

2026-10-08 09:53:50,637 - INFO - Connected to database: bts_flights_db


# 2. Load narrowed checkpoint

Loading the Parquet file skips the download and the column selection from Notebook 01.

In [2]:
CHECKPOINT_PATH = Path("checkpoints") / "df_narrowed_2024_bts.parquet"

df = pd.read_parquet(CHECKPOINT_PATH)
logger.info(f"Loaded narrowed checkpoint: {df.shape}")
df.head()

2026-10-08 09:53:54,683 - INFO - Loaded narrowed checkpoint: (7079061, 37)


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Quarter,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,...,ActualElapsedTime,AirTime,TaxiOut,TaxiIn,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2024,1,8,1,2024-01-08,1,9E,N485PX,4801.0,LGA,...,213.0,184.0,25.0,4.0,1148.0,NaN,NaN,NaN,NaN,NaN
1,2024,1,9,2,2024-01-09,1,9E,N912XJ,4801.0,LGA,...,196.0,168.0,16.0,12.0,1148.0,NaN,NaN,NaN,NaN,NaN
2,2024,1,10,3,2024-01-10,1,9E,N918XJ,4801.0,LGA,...,200.0,177.0,17.0,6.0,1148.0,NaN,NaN,NaN,NaN,NaN
3,2024,1,11,4,2024-01-11,1,9E,N490PX,4801.0,LGA,...,223.0,188.0,27.0,8.0,1148.0,0.0,0.0,4.0,0.0,23.0
4,2024,1,12,5,2024-01-12,1,9E,N915XJ,4801.0,LGA,...,226.0,185.0,24.0,17.0,1148.0,NaN,NaN,NaN,NaN,NaN


Same shape Notebook 01 saved (7,079,061 × 37), so the Parquet round-trip lost nothing.

# 3. Staging schema design

BTS has no per-record ID. A flight can only be identified by the combination of its business columns (date, carrier, flight number, origin, destination, scheduled times), so the primary key is a `row_hash CHAR(64)`: a SHA-256 over all business columns. With `ON CONFLICT (row_hash) DO NOTHING`, re-running the load skips rows that are already in the table.

Some type decisions:

- BTS stores times as HHMM integers (`1430` is 2:30 PM). I left them as `SMALLINT`. Nothing in the analysis needs a full timestamp, and the scheduled hour comes out of an integer division (`crs_dep_time / 100`).
- Delay and cause columns are nullable `REAL`, since their nulls are structural (Notebook 01, Section 4.1).
- Airport, state and tail-number codes are bounded `VARCHAR`s sized to BTS's documented formats.

In [3]:
staging_ddl = """
CREATE SCHEMA IF NOT EXISTS staging;

CREATE TABLE IF NOT EXISTS staging.stg_bts_flights (
    row_hash                CHAR(64) PRIMARY KEY,

    flight_year              SMALLINT NOT NULL,
    flight_month              SMALLINT NOT NULL,
    day_of_month              SMALLINT NOT NULL,
    day_of_week                SMALLINT NOT NULL,
    flight_date               DATE NOT NULL,
    quarter                   SMALLINT NOT NULL,

    reporting_airline          VARCHAR(4) NOT NULL,
    tail_number                VARCHAR(10),
    flight_number               INTEGER,

    origin                    VARCHAR(4) NOT NULL,
    origin_city_name           VARCHAR(60),
    origin_state                VARCHAR(2),

    dest                      VARCHAR(4) NOT NULL,
    dest_city_name             VARCHAR(60),
    dest_state                  VARCHAR(2),

    crs_dep_time                SMALLINT,
    dep_time                   SMALLINT,
    dep_delay                  REAL,
    dep_del15                   SMALLINT,

    crs_arr_time                SMALLINT,
    arr_time                    SMALLINT,
    arr_delay                   REAL,
    arr_del15                   SMALLINT,

    cancelled                  SMALLINT NOT NULL,
    cancellation_code           VARCHAR(1),
    diverted                   SMALLINT NOT NULL,

    crs_elapsed_time             REAL,
    actual_elapsed_time           REAL,
    air_time                    REAL,

    taxi_out                   REAL,
    taxi_in                    REAL,
    distance                   REAL,

    carrier_delay                REAL,
    weather_delay                REAL,
    nas_delay                   REAL,
    security_delay                REAL,
    late_aircraft_delay            REAL,

    loaded_at                   TIMESTAMP NOT NULL DEFAULT now()
);
"""

with engine.connect() as conn:
    conn.execute(text(staging_ddl))
    conn.commit()
    logger.info("staging.stg_bts_flights created (or already existed).")

2026-10-08 09:53:54,754 - INFO - staging.stg_bts_flights created (or already existed).


# 4. Data cleaning and transformation

An explicit rename map from BTS's CamelCase to snake_case keeps the correspondence with the DDL auditable in one place.

In [4]:
column_rename_map = {
    "Year": "flight_year",
    "Month": "flight_month",
    "DayofMonth": "day_of_month",
    "DayOfWeek": "day_of_week",
    "FlightDate": "flight_date",
    "Quarter": "quarter",
    "Reporting_Airline": "reporting_airline",
    "Tail_Number": "tail_number",
    "Flight_Number_Reporting_Airline": "flight_number",
    "Origin": "origin",
    "OriginCityName": "origin_city_name",
    "OriginState": "origin_state",
    "Dest": "dest",
    "DestCityName": "dest_city_name",
    "DestState": "dest_state",
    "CRSDepTime": "crs_dep_time",
    "DepTime": "dep_time",
    "DepDelay": "dep_delay",
    "DepDel15": "dep_del15",
    "CRSArrTime": "crs_arr_time",
    "ArrTime": "arr_time",
    "ArrDelay": "arr_delay",
    "ArrDel15": "arr_del15",
    "Cancelled": "cancelled",
    "CancellationCode": "cancellation_code",
    "Diverted": "diverted",
    "CRSElapsedTime": "crs_elapsed_time",
    "ActualElapsedTime": "actual_elapsed_time",
    "AirTime": "air_time",
    "TaxiOut": "taxi_out",
    "TaxiIn": "taxi_in",
    "Distance": "distance",
    "CarrierDelay": "carrier_delay",
    "WeatherDelay": "weather_delay",
    "NASDelay": "nas_delay",
    "SecurityDelay": "security_delay",
    "LateAircraftDelay": "late_aircraft_delay",
}

df = df.rename(columns=column_rename_map)
logger.info(f"Columns renamed. Current shape: {df.shape}")
df.dtypes

2026-10-08 09:53:55,317 - INFO - Columns renamed. Current shape: (7079061, 37)


flight_year              int16
flight_month              int8
day_of_month              int8
day_of_week               int8
flight_date             object
quarter                   int8
reporting_airline       object
tail_number             object
flight_number          float32
origin                  object
origin_city_name        object
origin_state            object
dest                    object
dest_city_name          object
dest_state              object
crs_dep_time             int16
dep_time               float32
dep_delay              float32
dep_del15              float32
crs_arr_time             int16
arr_time               float32
arr_delay              float32
arr_del15              float32
cancelled              float32
cancellation_code       object
diverted               float32
crs_elapsed_time       float32
actual_elapsed_time    float32
air_time               float32
taxi_out               float32
taxi_in                float32
distance               float32
carrier_

## 4.1 Surrogate key

pandas nulls (`NaN`, `NaT`, `pd.NA`) don't stringify consistently, so hashing them as they come could give different hashes for the same missing value. Every null becomes the literal string `"NULL"` before hashing, which keeps cancelled flights (null delays, null taxi times) reproducible.

In [5]:
business_columns = [col for col in df.columns if col != "row_hash"]

def compute_row_hash(row):
    parts = []
    for col in business_columns:
        val = row[col]
        parts.append("NULL" if pd.isna(val) else str(val))
    combined = "|".join(parts)
    return hashlib.sha256(combined.encode("utf-8")).hexdigest()

df["row_hash"] = df.apply(compute_row_hash, axis=1)
logger.info(f"row_hash computed. Unique hashes: {df['row_hash'].nunique():,} / {len(df):,} rows")

2026-10-08 10:08:52,993 - INFO - row_hash computed. Unique hashes: 7,079,061 / 7,079,061 rows


The hash took between 7 and 16 minutes in my runs, since it goes row by row in Python. There are 7,079,061 unique hashes for 7,079,061 rows, so the source has no fully duplicated records and the key works as a primary key.

There's one limit to this key. The hash depends on how pandas turns each value into text, so a change of dtype (a `float32` read as `float64`, for example) would give the same flight a different hash. The same happens if BTS revises a month after publishing it. In either case `ON CONFLICT` wouldn't recognize the flight and would insert it a second time. For a one-off load of 2024 that's acceptable. An incremental pipeline would need a check on a natural key as well, such as date, carrier, flight number, origin and scheduled departure.

# 5. Idempotent load to PostgreSQL

Rows go in batches of 50,000 through `execute_values`. psycopg2 can't adapt pandas null types (it raises `ProgrammingError: can't adapt type 'NAType'`), so `_sql_safe()` converts them to Python `None`, which becomes SQL `NULL`.

In [6]:
def _sql_safe(val):
    return None if pd.isna(val) else val

insert_columns = list(df.columns)  # includes row_hash
insert_sql = f"""
INSERT INTO staging.stg_bts_flights ({", ".join(insert_columns)})
VALUES %s
ON CONFLICT (row_hash) DO NOTHING
"""

BATCH_SIZE = 50_000
total_rows = len(df)
conn = psycopg2.connect(
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
)
cursor = conn.cursor()

for start in tqdm(range(0, total_rows, BATCH_SIZE), desc="Loading to Postgres"):
    batch = df.iloc[start:start + BATCH_SIZE]
    values = [
        tuple(_sql_safe(v) for v in row)
        for row in batch[insert_columns].itertuples(index=False, name=None)
    ]
    execute_values(cursor, insert_sql, values)
    conn.commit()

cursor.close()
conn.close()
logger.info("Load complete.")

Loading to Postgres: 100%|██████████| 142/142 [50:49<00:00, 21.48s/it]
2026-10-08 10:59:43,423 - INFO - Load complete.


142 batches, no errors. The first load took about 12 minutes. Re-running it over a table that already held all 7M rows took between 42 and 51 minutes in my runs, because every single row goes through the conflict check against the primary key.

That re-run was a useful accident: it's a real test of `ON CONFLICT (row_hash) DO NOTHING`. Section 6 shows whether the table doubled.

# 6. Validation

A partial failure or a silent type coercion wouldn't announce itself. This checks the row count against the DataFrame, looks for nulls in columns that must always be filled, and pulls a sample.

In [7]:
with engine.connect() as conn:
    db_row_count = conn.execute(text("SELECT COUNT(*) FROM staging.stg_bts_flights")).scalar()
    logger.info(f"Rows in Postgres: {db_row_count:,} | Rows in DataFrame: {len(df):,}")

    null_check = conn.execute(text("""
        SELECT
            COUNT(*) FILTER (WHERE reporting_airline IS NULL) AS null_airline,
            COUNT(*) FILTER (WHERE flight_date IS NULL) AS null_date,
            COUNT(*) FILTER (WHERE origin IS NULL) AS null_origin,
            COUNT(*) FILTER (WHERE dest IS NULL) AS null_dest
        FROM staging.stg_bts_flights
    """)).fetchone()
    logger.info(f"Unexpected nulls: {null_check}")

    sample = pd.read_sql(text("SELECT * FROM staging.stg_bts_flights LIMIT 5"), conn)

sample

2026-10-08 10:59:45,517 - INFO - Rows in Postgres: 7,079,061 | Rows in DataFrame: 7,079,061
2026-10-08 10:59:47,432 - INFO - Unexpected nulls: (0, 0, 0, 0)


,row_hash,flight_year,flight_month,day_of_month,day_of_week,flight_date,quarter,reporting_airline,tail_number,flight_number,...,air_time,taxi_out,taxi_in,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,loaded_at
0,bce7c4ec7b2a5d0b8702f95d9c6abecdf4b3b076c1b5d4...,2024,1,8,1,2024-01-08,1,9E,N485PX,4801,...,184.0,25.0,4.0,1148.0,NaN,NaN,NaN,NaN,NaN,2026-10-06 11:26:22.863987
1,8c8fbec5a8446f228a77fd290124a086024875919fdca5...,2024,1,9,2,2024-01-09,1,9E,N912XJ,4801,...,168.0,16.0,12.0,1148.0,NaN,NaN,NaN,NaN,NaN,2026-10-06 11:26:22.863987
2,fa0a643a3c13140d920516f831859297480af0be4c52bc...,2024,1,10,3,2024-01-10,1,9E,N918XJ,4801,...,177.0,17.0,6.0,1148.0,NaN,NaN,NaN,NaN,NaN,2026-10-06 11:26:22.863987
3,ab4be9f3559a7e7997944aac77ca42cc12c57462e322af...,2024,1,11,4,2024-01-11,1,9E,N490PX,4801,...,188.0,27.0,8.0,1148.0,0.0,0.0,4.0,0.0,23.0,2026-10-06 11:26:22.863987
4,a97f5ef50420513c4092e05b92a9623095f7bbb7a6b323...,2024,1,12,5,2024-01-12,1,9E,N915XJ,4801,...,185.0,24.0,17.0,1148.0,NaN,NaN,NaN,NaN,NaN,2026-10-06 11:26:22.863987


7,079,061 rows in PostgreSQL, the same as the DataFrame, even after loading the full year a second time on top of the first. Not one row was duplicated. There are no nulls in `reporting_airline`, `flight_date`, `origin` or `dest`, and the sample shows dates, carrier codes and tail numbers landing with the expected types.

# 7. Conclusions

The 2024 data now sits in `staging.stg_bts_flights`: 7,079,061 rows matching the source, with no nulls in carrier, date, origin or destination. The `row_hash` key had no collisions, and loading the whole year twice still left exactly 7,079,061 rows, so the notebook can be re-run without side effects.

Notebook 03 queries this table directly.